# 2608.27372 — Ellipsoid-Fitting Phase Boundary

**Engineering Statements companion notebook — v2**

This version is designed so **Run all** produces visible numerical output and a transition plot.

\[
\text{asymptotic specification}
\rightarrow
\text{computation}
\rightarrow
\text{reading}
\]

Source: arXiv:2608.27372  
Engineering Statement: `statements/2608-27372.yaml`

## 1. Runtime mode

`fast` is the default so the notebook produces results quickly in Colab.

Change `MODE` to `"paper"` to run the larger \(d=40\), 8-trial sweep used for the report-scale comparison.

In [ ]:
MODE = "fast"   # "fast" or "paper"

if MODE == "fast":
    D = 12
    ALPHAS = [0.10, 0.15, 0.20, 0.23, 0.25, 0.27, 0.30, 0.35, 0.40]
    TRIALS = 3
else:
    D = 40
    ALPHAS = [0.10, 0.15, 0.20, 0.225, 0.24, 0.25, 0.26, 0.275, 0.30, 0.35, 0.40]
    TRIALS = 8

SEED = 260827372

print({
    "mode": MODE,
    "d": D,
    "trials_per_density": TRIALS,
    "alpha_values": ALPHAS,
})

## 2. Install and import dependencies

In [ ]:
!pip -q install pyyaml cvxpy

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import yaml

print("cvxpy:", cp.__version__)
print("solvers:", cp.installed_solvers())

## 3. Load the Engineering Statement

The repository YAML is primary. An embedded fallback keeps the notebook portable.

In [ ]:
STATEMENT_PATH = Path("../statements/2608-27372.yaml")

fallback_yaml = """
id: 2608-27372
title: Universality and Sharp Thresholds for Ellipsoid Fitting
source:
  paper: https://arxiv.org/abs/2608.27372
objective: >
  Specify the ellipsoid-fitting problem through its constraint-density scaling,
  sharp SAT-UNSAT phase boundary, distribution-dependent threshold,
  and computational readings.
constraints:
  - Random vectors x_1,...,x_n lie in R^d.
  - Seek a positive-semidefinite matrix R such that x_i^T R x_i = 1 for every i.
  - Constraint density is alpha = n / d^2.
  - A sharp SAT-UNSAT phase boundary occurs at alpha_star(kappa).
  - The coordinate distribution enters the phase boundary through kappa = E[x_ij^4].
  - For Gaussian coordinates, kappa = 3 and alpha_star(3) = 1/4.
"""

if STATEMENT_PATH.exists():
    statement = yaml.safe_load(STATEMENT_PATH.read_text())
    print("Loaded:", STATEMENT_PATH)
else:
    statement = yaml.safe_load(fallback_yaml)
    print("Using embedded fallback statement.")

print(statement["title"])
print(statement["objective"].strip())

## 4. Gaussian specification

For Gaussian coordinates,

\[
\kappa=3,
\qquad
\alpha_\star(3)=\frac14.
\]

For \(d=40\), the asymptotic boundary corresponds to

\[
n \approx \frac{40^2}{4}=400.
\]

In [ ]:
KAPPA_GAUSSIAN = 3.0
ALPHA_STAR = 1.0 / 4.0

print("Gaussian specification")
print("kappa =", KAPPA_GAUSSIAN)
print("alpha_star =", ALPHA_STAR)
print("d = 40 -> n ≈", int(ALPHA_STAR * 40**2))

## 5. Generate Gaussian instances

In [ ]:
def gaussian_instance(d, n, rng):
    return rng.normal(size=(n, d))

## 6. Ellipsoid-fitting SDP

For each random instance, solve for a symmetric matrix \(R\) satisfying

\[
x_i^\top R x_i = 1
\]

for every sampled point, together with the positive-semidefinite matrix constraint.

The solver result is recorded as a computational **SAT** or **UNSAT** reading.

In [ ]:
def choose_solver():
    installed = set(cp.installed_solvers())
    for candidate in ("CLARABEL", "SCS"):
        if candidate in installed:
            return candidate
    raise RuntimeError("Neither CLARABEL nor SCS is available.")

SOLVER = choose_solver()
print("Using solver:", SOLVER)

def ellipsoid_fit_reading(X, solver=SOLVER):
    X = np.asarray(X, dtype=float)
    n, d = X.shape

    R = cp.Variable((d, d), symmetric=True)
    constraints = [R >> 0]
    constraints += [cp.quad_form(X[i], R) == 1 for i in range(n)]

    problem = cp.Problem(cp.Minimize(0), constraints)

    try:
        if solver == "SCS":
            problem.solve(solver=solver, verbose=False, eps=1e-5, max_iters=20000)
        else:
            problem.solve(solver=solver, verbose=False)
    except Exception as exc:
        return "ERROR", type(exc).__name__

    feasible = problem.status in {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}
    return ("SAT" if feasible else "UNSAT"), problem.status

## 7. Quick validation reading

This cell always produces a visible result before the density sweep.

In [ ]:
rng = np.random.default_rng(SEED)
X_check = gaussian_instance(d=6, n=6, rng=rng)
reading, status = ellipsoid_fit_reading(X_check)

print({
    "d": 6,
    "n": 6,
    "alpha": 6 / 6**2,
    "reading": reading,
    "solver_status": status,
})

## 8. Sweep constraint density

At each value of

\[
\alpha=\frac{n}{d^2},
\]

the notebook generates repeated random instances and computes the **fraction SAT**.

In [ ]:
def sweep_gaussian(d, alphas, trials, seed):
    rng = np.random.default_rng(seed)
    rows = []

    total = len(alphas) * trials
    completed = 0

    for alpha_target in alphas:
        n = max(1, int(round(alpha_target * d**2)))
        sat_count = 0
        error_count = 0

        for trial in range(trials):
            X = gaussian_instance(d=d, n=n, rng=rng)
            reading, status = ellipsoid_fit_reading(X)

            sat_count += int(reading == "SAT")
            error_count += int(reading == "ERROR")
            completed += 1
            print(
                f"{completed:>3}/{total}  "
                f"d={d:>2} n={n:>4} alpha={n/d**2:.4f}  "
                f"trial={trial+1}/{trials}  {reading}"
            )

        rows.append({
            "d": d,
            "n": n,
            "alpha": n / d**2,
            "trials": trials,
            "sat_count": sat_count,
            "fraction_sat": sat_count / trials,
            "error_count": error_count,
        })

    return pd.DataFrame(rows)

readings = sweep_gaussian(
    d=D,
    alphas=ALPHAS,
    trials=TRIALS,
    seed=SEED,
)

readings

## 9. Computational readings table

In [ ]:
display(
    readings[
        ["d", "n", "alpha", "trials", "sat_count", "fraction_sat", "error_count"]
    ]
)

## 10. Plot the computational transition

The dashed vertical reference is the Gaussian asymptotic specification

\[
lpha_\star(3)=1/4.
\]

The plotted points are computational readings.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    readings["alpha"],
    readings["fraction_sat"],
    marker="o",
    linewidth=2,
    label="Computational fraction SAT",
)
ax.axvline(
    ALPHA_STAR,
    linestyle="--",
    linewidth=2,
    label=r"Theoretical boundary $\alpha_\star(3)=1/4$",
)

ax.set_xlabel(r"Constraint density $\alpha=n/d^2$")
ax.set_ylabel("Fraction SAT")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Ellipsoid-fitting readings — d={D}, {TRIALS} trials per density")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 11. Save outputs

This cell creates repository-ready output artifacts every time the notebook is run.

In [ ]:
OUTPUT_DIR = Path("../outputs/2608-27372")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_readings.csv"
png_path = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_transition.png"

readings.to_csv(csv_path, index=False)

fig.savefig(
    png_path,
    dpi=180,
    bbox_inches="tight",
)

print("Saved:")
print(csv_path)
print(png_path)

## 12. Reading

The theoretical and computational objects remain distinct:

\[
\boxed{\alpha_\star(\kappa)}
\]

specifies the asymptotic phase boundary, while the SDP sweep produces computational readings at chosen \(d\) and \(n\).

For the Gaussian case:

\[
\boxed{
\kappa=3
\rightarrow
\alpha_\star(3)=\frac14
\rightarrow
\text{computation}
\rightarrow
\text{reading}
}
\]

To run the report-scale experiment, change:

```python
MODE = "paper"
```

and run all cells again.